# 04 — OCR Batch Processing

Batch-process collected PDF financial reports through OCR pipelines
(PaddleOCR / Qwen3-VL) to produce page images + extracted text for
training zone classification and table extraction models.

**Input**: PDF reports from notebooks 01 & 02
**Output**: Page images + OCR text in data/ocr_output/

In [ ]:
# Cell 1: Setup
!pip install -q paddlepaddle paddleocr PyMuPDF Pillow tqdm PyYAML

import json, os
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import numpy as np
import yaml

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

CFG_PATH = Path('../configs/training_config.yaml')
if not CFG_PATH.exists():
    CFG_PATH = Path('/content/drive/MyDrive/numera-ml/configs/training_config.yaml')
cfg = yaml.safe_load(CFG_PATH.read_text())

DATA_DIR = Path(cfg['drive']['data_dir'])
OCR_DIR = DATA_DIR / 'ocr_output'
IMG_DIR = OCR_DIR / 'images'
OCR_DIR.mkdir(parents=True, exist_ok=True)
IMG_DIR.mkdir(parents=True, exist_ok=True)

DPI = cfg['ocr']['dpi']
BATCH_SIZE = cfg['ocr']['batch_size']
print(f'DPI: {DPI}, batch: {BATCH_SIZE}')

In [ ]:
# Cell 2: Initialize PaddleOCR engine
from paddleocr import PaddleOCR

ocr_engine = PaddleOCR(use_angle_cls=True, lang='en', show_log=False)
print('PaddleOCR initialized ✅')

In [ ]:
# Cell 3: Convert PDF pages to images
import fitz  # PyMuPDF

def pdf_to_images(pdf_path: Path, dpi: int = 300) -> list[tuple[int, Image.Image]]:
    """Convert PDF pages to PIL Images."""
    doc = fitz.open(str(pdf_path))
    images = []
    zoom = dpi / 72
    mat = fitz.Matrix(zoom, zoom)
    for i, page in enumerate(doc):
        pix = page.get_pixmap(matrix=mat)
        img = Image.frombytes('RGB', (pix.width, pix.height), pix.samples)
        images.append((i + 1, img))
    doc.close()
    return images

# Collect all PDFs
pdf_files = list(DATA_DIR.glob('edgar/*.html'))  # EDGAR HTML (converted)
pdf_files += list(DATA_DIR.glob('lse/*.pdf'))
pdf_files += list(DATA_DIR.glob('gcc/*.pdf'))
print(f'Total PDFs to process: {len(pdf_files)}')

In [ ]:
# Cell 4: Batch OCR processing
ocr_results = []

for pdf_path in tqdm(pdf_files, desc='OCR batch'):
    if not pdf_path.suffix == '.pdf':
        continue
    try:
        page_images = pdf_to_images(pdf_path, dpi=DPI)
    except Exception as e:
        print(f'  Error converting {pdf_path.name}: {e}')
        continue
    
    doc_result = {
        'file': pdf_path.name,
        'total_pages': len(page_images),
        'pages': [],
    }
    
    for page_num, img in page_images:
        # Save page image
        img_name = f"{pdf_path.stem}_p{page_num:03d}.png"
        img_path = IMG_DIR / img_name
        if not img_path.exists():
            img.save(str(img_path))
        
        # Run OCR
        try:
            img_array = np.array(img)
            result = ocr_engine.ocr(img_array, cls=True)
            blocks = []
            if result and result[0]:
                for line in result[0]:
                    bbox = line[0]  # [[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
                    text = line[1][0]
                    conf = line[1][1]
                    blocks.append({
                        'text': text,
                        'confidence': round(conf, 4),
                        'bbox': [bbox[0][0], bbox[0][1], bbox[2][0], bbox[2][1]],
                    })
            doc_result['pages'].append({
                'page': page_num,
                'image': img_name,
                'blocks': blocks,
                'full_text': ' '.join(b['text'] for b in blocks),
            })
        except Exception as e:
            print(f'  OCR error {pdf_path.name} p{page_num}: {e}')
            doc_result['pages'].append({'page': page_num, 'image': img_name, 'blocks': [], 'full_text': ''})
    
    ocr_results.append(doc_result)

print(f'Processed {len(ocr_results)} documents')

In [ ]:
# Cell 5: Save results and statistics
out_file = OCR_DIR / 'ocr_results.json'
with open(out_file, 'w') as f:
    json.dump(ocr_results, f, indent=2)

total_pages = sum(r['total_pages'] for r in ocr_results)
total_blocks = sum(len(p['blocks']) for r in ocr_results for p in r['pages'])
avg_conf = np.mean([b['confidence'] for r in ocr_results for p in r['pages'] for b in p['blocks']]) if total_blocks > 0 else 0

print(f'OCR Results:')
print(f'  Documents: {len(ocr_results)}')
print(f'  Pages: {total_pages}')
print(f'  Text blocks: {total_blocks}')
print(f'  Avg confidence: {avg_conf:.4f}')
print(f'  Images saved: {len(list(IMG_DIR.glob("*.png")))}')
print(f'  Output: {out_file}')